In [ ]:
from getpass import getpass

admin_rdm_url = 'https://admin.staging.rdm.example.com/'
cas_rdm_url = 'https://account.staging.rdm.example.com/'

idp_name_integrated_admin = None
idp_username_integrated_admin = None
idp_password_integrated_admin = None
idp_username_integrated_admin_2 = None
idp_password_integrated_admin_2 = None

user_guid = 'abcde'
user_name = 'test@example.com'
integrated_admin_user_name = 'admin@example.com'
registration_provider_name = 'OSF Registries'
preprint_provider_name = 'Open Science Framework'
collection_provider_name = 'OSF Collection'

default_result_path = None
close_on_fail = False
transition_timeout = 60000

In [ ]:
if idp_name_integrated_admin is None:
    idp_name_integrated_admin = input(prompt=f'IdP Name for integrated admin: ')
if idp_username_integrated_admin is None:
    idp_username_integrated_admin = input(prompt=f'Username for {idp_name_integrated_admin}')
if idp_password_integrated_admin is None:
    idp_password_integrated_admin = getpass(prompt=f'Password for {idp_username_integrated_admin}@{idp_name_integrated_admin}')
if idp_username_integrated_admin_2 is None:
    idp_username_integrated_admin_2 = input(prompt=f'Username for {idp_name_integrated_admin} (統合管理者2)')
if idp_password_integrated_admin_2 is None:
    idp_password_integrated_admin_2 = getpass(prompt=f'Password for {idp_username_integrated_admin_2}@{idp_name_integrated_admin} (統合管理者2)')

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# 統合管理者のアクセス制御を確認する

- サブシステム名: 管理者
- ページ/アドオン: 管理者
- 機能分類: 認証/認可
- シナリオ名: 統合管理者のアクセス制御を確認する
- 用意するテストデータ:   
   ・GRDM 管理者のURL   
   ・CASのURL   
   ・アカウント(統合管理者）   
   ・アカウント(統合管理者2）   
   ・ユーザーのGUID（削除されていないユーザー）   
   ・ユーザー名（削除されていないユーザー）   
   ・アカウント(統合管理者）のユーザー名    
   ・登録プロバイダー(OSF Registries、少なくとも1件のRegistrationが紐付いている)   
   ・プレプリントプロバイダー(Open Science Framework、少なくとも1件のプレプリントが紐付いている)   
   ・コレクションプロバイダー(OSF Collection、少なくとも1件のプロジェクトが紐付いている)   
   ・USE_LOGIN_FORM が空（ENABLE_LOGIN_FORM が偽）の環境

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGakuNin RDM管理者のトップページを表示する

・`/account/login/` にリダイレクトされ、ログイン画面が表示されること   
・リダイレクト後のURLが「https://{admin_domain}/account/login/」であること

In [ ]:
import re
from urllib.parse import urlparse

async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

    login_url = page.url
    parsed_url = urlparse(login_url)
    assert re.fullmatch(
        r"/account/login/",
        parsed_url.path,
    ), f"リダイレクト後のURLが期待する形式ではありません。actual: {login_url}"

await run_pw(_step)

## アカウント(統合管理者)を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。


In [ ]:
csrf_token = None

async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_integrated_admin, idp_username_integrated_admin, idp_password_integrated_admin, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

    cookies = await page.context.cookies()
    global csrf_token
    csrf_token = next((c['value'] for c in cookies if c['name'] == 'admin-csrf'), None)

await run_pw(_step)

## 「登録プロバイダー」メニューから「リスト」を選択する

「List of Registration Providers」ページが表示されること   
※ 画面見出し: List of Registration Providers

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "#collapseRegistrationProviders"]').click()
    await page.locator('//a[@href = "/registration_providers/"]').click()
    
    await expect(page.locator('//h2[text() = "List of Registration Providers"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「List of Registration Providers」ページで、「OSF Registries」を選択する

・「登録プロバイダー詳細」ページが表示されること   
・「Change allowed schemas」ボタンが表示されること   
・「Delete registration provider」ボタンが表示されること   
・リダイレクト後のURLが「https://{admin_domain}/registration_providers/{registration_provider_id}/」であること   
※ 画面見出し: OSF Registries


In [ ]:
registration_provider_url = None

async def _step(page):
    await page.get_by_role("link", name=f'{registration_provider_name}').click()

    await expect(page.locator(f'//h2[text() = "{registration_provider_name}"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="Change allowed schemas")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="Delete registration provider")).to_be_visible(timeout=transition_timeout)
    
    global registration_provider_url
    registration_provider_url = page.url
    parsed_url = urlparse(registration_provider_url)
    assert re.fullmatch(
        r"/registration_providers/\d+/",
        parsed_url.path,
    ), f"リダイレクト後のURLが期待する形式ではありません。actual: {registration_provider_url}"

await run_pw(_step)

## 「Change allowed schemas」ボタンをクリックする	

・「OSF Registries」ページが表示されること   
・リダイレクト後のURLが「https://{admin_domain}/registration_providers/{registration_provider_id}/schemas/」であること   
※ 画面見出し: OSF Registries

In [ ]:
import os

url_list_file = os.path.join(default_result_path, 'url_list.txt')
schema_was_checked_before_test = None

async def _step(page):
    await page.get_by_role("link", name="Change allowed schemas").click()
    
    await expect(page.locator(f'//h2[text() = "{registration_provider_name}"]')).to_be_visible(timeout=transition_timeout)

    registration_provider_schemas_url = page.url
    parsed_url = urlparse(registration_provider_schemas_url)
    assert re.fullmatch(
        r"/registration_providers/\d+/schemas/",
        parsed_url.path,
    ), f"リダイレクト後のURLが期待する形式ではありません。actual: {registration_provider_schemas_url}"
        
    # Save registration_provider_schemas_url to a file
    with open(url_list_file, 'a') as f:
        f.write(f'registration_provider_schemas_url={registration_provider_schemas_url}\n')

    schema_row = page.locator("tr").filter(has_text="Confirmatory - General v2")
    global schema_was_checked_before_test
    schema_was_checked_before_test = await schema_row.locator('input[type="checkbox"]').is_checked()

    print(f'Saved URLs to {url_list_file}')


await run_pw(_step)

## 「Confirmatory - General v2」のチェックボックスをクリックし、「Update Schemas」ボタンをクリックする

・「登録プロバイダー詳細」ページが表示されること   
・「Change allowed schemas」ボタンが表示されること   
・「Delete registration provider」ボタンが表示されること   
・リダイレクト後のURLが「https://{admin_domain}/registration_providers/{registration_provider_id}/」であること   
※ 画面見出し: OSF Registries

In [ ]:
async def _step(page):
    schema_row = page.locator("tr").filter(has_text="Confirmatory - General v2")
    await schema_row.locator('input[type="checkbox"]').check()
    await page.get_by_role("button", name="Update Schemas").click()
    
    await expect(page.locator(f'//h2[text() = "{registration_provider_name}"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="Change allowed schemas")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="Delete registration provider")).to_be_visible(timeout=transition_timeout)
    
    global registration_provider_url
    registration_provider_url = page.url
    parsed_url = urlparse(registration_provider_url)
    assert re.fullmatch(
        r"/registration_providers/\d+/",
        parsed_url.path,
    ), f"リダイレクト後のURLが期待する形式ではありません。actual: {registration_provider_url}"

await run_pw(_step)

## 「OSF Registries」の「Change allowed schemas」画面を開き、「Confirmatory - General v2」の選択状態をテスト実施前の状態に戻す。「Update Schemas」ボタンをクリックする。

・「登録プロバイダー詳細」ページが表示されること   
・「Change allowed schemas」ボタンが表示されること   
・「Delete registration provider」ボタンが表示されること   

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="Change allowed schemas").click()
    
    await expect(page.locator(f'//h2[text() = "{registration_provider_name}"]')).to_be_visible(timeout=transition_timeout)

    schema_row = page.locator("tr").filter(has_text="Confirmatory - General v2")
    if schema_was_checked_before_test:
        await schema_row.locator('input[type="checkbox"]').check()
    else:
        await schema_row.locator('input[type="checkbox"]').uncheck()

    await page.get_by_role("button", name="Update Schemas").click()
    
    await expect(page.locator(f'//h2[text() = "{registration_provider_name}"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="Change allowed schemas")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="Delete registration provider")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「Delete registration provider」ボタンをクリックする	

・「Cannot Delete OSF Registries」ページが表示されること   
・リダイレクト後のURLが「https://{admin_domain}/registration_providers/{registration_provider_id}/cannot_delete/」であること  
※ 画面見出し: Cannot Delete OSF Registries

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="Delete registration provider").click()
    
    await expect(page.locator(f'//h3[text() = "Cannot Delete {registration_provider_name}"]')).to_be_visible(timeout=transition_timeout)

    registration_provider_cannot_delete_url = page.url
    parsed_url = urlparse(registration_provider_cannot_delete_url)
    assert re.fullmatch(
        r"/registration_providers/\d+/cannot_delete/",
        parsed_url.path,
    ), f"リダイレクト後のURLが期待する形式ではありません。actual: {registration_provider_cannot_delete_url}"
        
    # Save registration_provider_cannot_delete_url to a file
    with open(url_list_file, 'a') as f:
        f.write(f'registration_provider_cannot_delete_url={registration_provider_cannot_delete_url}\n')

    print(f'Saved URLs to {url_list_file}')

await run_pw(_step)

## ブラウザのアドレスバーに、存在しない登録プロバイダーのIDを「Cannot Delete OSF Registries」ページのURLに入力する

404ページが表示されること   
「No RegistrationProvider matches the given query.」というテキストが表示されること

In [ ]:
not_found_registration_provider_id = 9999
not_found_status = 404

async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/registration_providers/{not_found_registration_provider_id}/cannot_delete")

    await expect(page.get_by_role("heading", name=f"{not_found_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text("No RegistrationProvider matches the given query.")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「プレプリントプロバイダー」メニューから「リスト」を選択する

「List of Preprint Providers」ページが表示されること   
※ 画面見出し: List of Preprint Providers

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "#collapsePreprintProviders"]').click()
    await page.locator('//a[@href = "/preprint_providers/"]').click()
    
    await expect(page.locator('//h2[text() = "List of Preprint Providers"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「List of Preprint Providers」ページで、「Open Science Framework」を選択する	

・「プレプリントプロバイダー詳細」ページが表示されること   
・「Delete preprint provider」ボタンが表示されること   
※ 画面見出し: Open Science Framework


In [ ]:
async def _step(page):
    await page.get_by_role("link", name=f'{preprint_provider_name}').click()

    await expect(page.locator(f'//h2[text() = "{preprint_provider_name}"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="Delete preprint provider")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「Delete preprint provider」ボタンをクリックする	

・「Cannot Delete Open Science Framework」ページが表示されること    
・リダイレクト後のURLが「https://{admin_domain}/preprint_providers/{preprint_provider_id}/cannot_delete/」であること   
※ 画面見出し: Cannot Delete Open Science Framework


In [ ]:
async def _step(page):
    await page.get_by_role("link", name="Delete preprint provider").click()

    await expect(page.locator(f'//h3[text() = "Cannot Delete {preprint_provider_name}"]')).to_be_visible(timeout=transition_timeout)

    preprint_provider_cannot_delete_url = page.url
    parsed_url = urlparse(preprint_provider_cannot_delete_url)
    assert re.fullmatch(
        r"/preprint_providers/\d+/cannot_delete/",
        parsed_url.path,
    ), f"リダイレクト後のURLが期待する形式ではありません。actual: {preprint_provider_cannot_delete_url}"
        
    # Save preprint_provider_cannot_delete_url to a file
    with open(url_list_file, 'a') as f:
        f.write(f'preprint_provider_cannot_delete_url={preprint_provider_cannot_delete_url}\n')

    print(f'Saved URLs to {url_list_file}')

await run_pw(_step)

## ブラウザのアドレスバーに、存在しないプレプリントプロバイダーのIDを「Cannot Delete Open Science Framework」ページのURLに入力する

404ページが表示されること   
「No PreprintProvider matches the given query.」というテキストが表示されること

In [ ]:
not_found_preprint_provider_id = 9999

async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/preprint_providers/{not_found_preprint_provider_id}/cannot_delete")

    await expect(page.get_by_role("heading", name=f"{not_found_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text("No PreprintProvider matches the given query.")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「コレクションプロバイダー」メニューから「リスト」を選択する	

「コレクションプロバイダのリスト」ページが表示されること   
※ 画面見出し: コレクションプロバイダのリスト

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "#collapseCollectionProviders"]').click()
    await page.locator('//a[@href = "/collection_providers/"]').click()
    
    await expect(page.locator('//h2[text() = "コレクションプロバイダのリスト"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「コレクションプロバイダのリスト」ページで、「OSF Collection」を選択する	

・「コレクションプロバイダー詳細」ページが表示されること   
・「コレクションプロバイダを削除」ボタンが表示されること   
※ 画面見出し: OSF Collection

In [ ]:
async def _step(page):
    await page.get_by_role("link", name=f'{collection_provider_name}').click()

    await expect(page.locator(f'//h2[text() = "{collection_provider_name}"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="コレクションプロバイダを削除")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「コレクションプロバイダを削除」ボタンをクリックする	

・「OSF Collectionを削除できません」ページが表示されること    
・リダイレクト後のURLが「https://{admin_domain}/collection_providers/{collection_provider_id}/cannot_delete/」であること   
※ 画面見出し: OSF Collectionを削除できません

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="コレクションプロバイダを削除").click()

    await expect(page.locator(f'//h3[text() = "{collection_provider_name}を削除できません"]')).to_be_visible(timeout=transition_timeout)

    collection_provider_cannot_delete_url = page.url
    parsed_url = urlparse(collection_provider_cannot_delete_url)
    assert re.fullmatch(
        r"/collection_providers/\d+/cannot_delete/",
        parsed_url.path,
    ), f"リダイレクト後のURLが期待する形式ではありません。actual: {collection_provider_cannot_delete_url}"
        
    # Save collection_provider_cannot_delete_url to a file
    with open(url_list_file, 'a') as f:
        f.write(f'collection_provider_cannot_delete_url={collection_provider_cannot_delete_url}\n')

    print(f'Saved URLs to {url_list_file}')

await run_pw(_step)

## ブラウザのアドレスバーに、存在しないコレクションプロバイダーのIDを「OSF Collectionを削除できません」ページのURLに入力する

404ページが表示されること   
「No CollectionProvider matches the given query.」というテキストが表示されること

In [ ]:
not_found_collection_provider_id = 9999

async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/collection_providers/{not_found_collection_provider_id}/cannot_delete")

    await expect(page.get_by_role("heading", name=f"{not_found_status}")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text("No CollectionProvider matches the given query.")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「Management Commands」メニューを選択する	

「Management Commands」ページが表示されること   
※ 画面見出し: Management Commands

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/management/"]').click()
    
    await expect(page.locator("//h1[normalize-space(.)='Management Commands']")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「Run」ボタンをクリックする	

「Management Commands」ページが表示されること   
※ 画面見出し: Management Commands

In [ ]:
async def _step(page):
    await page.get_by_role("button", name="Run").click()
    
    await expect(page.locator("//h1[normalize-space(.)='Management Commands']")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

##  ブラウザのアドレスバーに、「https://{admin_domain}/statistics/index」を直接入力する	

「利用統計」ページが表示されること   
※ 画面見出し: 利用統計

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/index")

    await expect(page.locator('//h2[text() = "利用統計"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/quota_recalc」を直接入力する	

以下のJSONデータが表示されること   
{"status": "OK", "message": "<件数> users' quota successfully recalculated!"}   

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/quota_recalc")

    response_text = await page.locator("body").inner_text()
    assert re.search(
        r'"status"\s*:\s*"OK"\s*,\s*"message"\s*:\s*"\d+ users\' quota successfully recalculated!"',
        response_text,
    ), f"期待するJSONデータが表示されていません。actual: {response_text}"

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/quota_recalc/{ユーザーのGUID}」を直接入力する

以下のJSONデータが表示されること   
{"status": "OK", "message": "User's quota successfully recalculated!"}

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/quota_recalc/{user_guid}")

    response_text = await page.locator("body").inner_text()
    assert re.search(
        r'"status"\s*:\s*"OK"\s*,\s*"message"\s*:\s*"User\'s quota successfully recalculated!"',
        response_text,
    ), f"期待するJSONデータが表示されていません。actual: {response_text}"

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/statistics/test/mail」を直接入力する	

機関名ごとの送信結果（JSON）が表示されること   
JSON内に"is_success"が表示されること

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/test/mail")

    response_text = await page.locator("body").inner_text()
    assert "is_success" in str(response_text), (f"期待するJSONデータが表示されていません。actual: {response_text}")

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/Shibboleth.sso/Status」を直接入力する

ページにアクセスできないこと   
「You don't have permission to access this resource.」というテキストが表示されること

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{admin_rdm_url.rstrip('/')}/Shibboleth.sso/Status")

    await expect(page.get_by_role("heading", name="Forbidden")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text("You don't have permission to access this resource.")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{cas_domain}/Shibboleth.sso/Status」を直接入力する

ページにアクセスできないこと   
「You don't have permission to access this resource.」というテキストが表示されること

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{cas_rdm_url.rstrip('/')}/Shibboleth.sso/Status")

    await expect(page.get_by_role("heading", name="Forbidden")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text("You don't have permission to access this resource.")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## コマンドラインで、「https://{admin_domain}/password/recover」に対してPOSTメソッドを実行する。

HTTPステータスコードが 404 であること   
レスポンスボディに「Resolver404」というテキストが含まれていること   
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される。

In [ ]:
not_found_url_message = "Resolver404"

async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    response = await page.request.post(
        f"{admin_rdm_url.rstrip('/')}/password/recover",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Referer": f"{admin_rdm_url.rstrip('/')}/password/recover",
        },
        form={
            "csrfmiddlewaretoken": csrf_token,
            "username": user_name,
        },
        max_redirects=0,
    )

    assert response.status == not_found_status, (
        f"Expected HTTP status 404, but got {response.status}"
    )

    response_body = await response.text()
    assert not_found_url_message in response_body, (
        f"Expected message '{not_found_url_message}' "
        f"not found in response body: {response_body}"
    )

await run_pw(_step)


## ブラウザのアドレスバーに、「https://{admin_domain}/password/recover/{signature}」を直接入力する

404ページが表示されること    
「Resolver404」というテキストが表示されること   
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される。

In [ ]:
signature = 'abcde'

async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{admin_rdm_url.rstrip('/')}/password/recover/{signature}")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/password/reset/{token}/」を直接入力する

404ページが表示されること    
「Resolver404」というテキストが表示されること   
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される。

In [ ]:
token = 'dummytoken'

async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{admin_rdm_url.rstrip('/')}/password/reset/{token}")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## コマンドラインで、「https://{admin_domain}/password/reset/{token}」に対してPOSTメソッドを実行する。

HTTPステータスコードが 404 であること   
レスポンスボディに「Resolver404」というテキストが含まれていること   
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    fake_token = 'abc123-invalidtoken0000000'
    response = await page.request.post(
        f"{admin_rdm_url.rstrip('/')}/password/reset/{fake_token}/",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Referer": f"{admin_rdm_url.rstrip('/')}/password/reset/{fake_token}/",
        },
        form={
            "csrfmiddlewaretoken": csrf_token,
            "password1": "TestPassword123!",
            "password2": "TestPassword123!",
        },
        max_redirects=0,
    )

    assert response.status == not_found_status, (
        f"Expected HTTP status 404, but got {response.status}"
    )

    response_body = await response.text()
    assert not_found_url_message in response_body, (
        f"Expected message '{not_found_url_message}' "
        f"not found in response body: {response_body}"
    )

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/password/reset/done」を直接入力する	

404ページが表示されること   
「Resolver404」というテキストが表示されること   
※ ENABLE_LOGIN_FORM が無効なら、このURLにアクセスすると404が返される

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    await page.goto(f"{admin_rdm_url.rstrip('/')}/password/reset/done")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/statistics/gather/{ts}/{signature}」を直接入力する

以下のJSONデータが表示されること   
{"state": "fail", "error": "access forbidden"}

In [ ]:
ts = 1

async def _step(page):
    print(f"{admin_rdm_url.rstrip('/')}/statistics/gather/{ts}/{signature}")
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/gather/{ts}/{signature}")

    response_text = await page.locator("body").inner_text()
    assert re.search(
        r'"state"\s*:\s*"fail"\s*,\s*"error"\s*:\s*"access forbidden"',
        response_text,
    ), f"期待するJSONデータが表示されていません。actual: {response_text}"

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/statistics/gather/{access_token}」を直接入力する	

404ページが表示されること   
「Resolver404」というテキストが表示されること   

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/gather/{token}")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/external_acc_update/{ts}/{signature}」を直接入力する

以下のJSONデータが表示されること   
{"state": "fail", "error": "access forbidden"}

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/external_acc_update/{ts}/{signature}")

    response_text = await page.locator("body").inner_text()
    assert re.search(
        r'"state"\s*:\s*"fail"\s*,\s*"error"\s*:\s*"access forbidden"',
        response_text,
    ), f"期待するJSONデータが表示されていません。actual: {response_text}"

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/custom_storage_location/external_acc_update/{access_token}」を直接入力する	

404ページが表示されること   
「Resolver404」というテキストが表示されること   

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/custom_storage_location/external_acc_update/{token}")

    await expect(page.locator("h1 strong")).to_have_text(f"{not_found_status}", timeout=transition_timeout)
    await expect(page.locator("h2")).to_have_text(not_found_url_message, timeout=transition_timeout)

await run_pw(_step)

## コマンドラインで、「https://{admin_domain}/account/login」に対してPOSTメソッドを実行する。

HTTPステータスコードが 302 であること   
レスポンスヘッダーに「Email and/or Password incorrect. Please try again.」というテキストが含まれていること   
※ 正しいメールアドレスとパスワードを送ってもログインせず、エラーメッセージ（「Email and/or Password incorrect. Please try again.」）付きでログイン画面へリダイレクトされます

In [ ]:
async def _step(page):
    if idp_name_integrated_admin == 'FakeCAS':
        return

    response = await page.request.post(
        f"{admin_rdm_url.rstrip('/')}/account/login",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Referer": f"{admin_rdm_url.rstrip('/')}/account/login",
        },
        form={
            "csrfmiddlewaretoken": csrf_token,
            "email": idp_username_integrated_admin,
            "password": idp_password_integrated_admin,
        },
        max_redirects=0,
    )

    assert response.status == 302, (f"Expected HTTP status 302, but got {response.status}")

    response_headers = str(response.headers)
    expected_message = "Email and/or Password incorrect. Please try again."
    assert expected_message in response_headers, (
        f"Expected message '{expected_message}' "
        f"not found in response headers: {response_headers}"
    )

await run_pw(_step)

## アカウント(統合管理者)をログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    logout_btn = page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link = page.locator('//*[@href="/account/logout/"]')
    if await logout_btn.count() > 0:
        await logout_btn.click()
    else:
        await logout_link.click()

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アカウント(統合管理者2)を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。   

In [ ]:
async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_integrated_admin, idp_username_integrated_admin_2, idp_password_integrated_admin_2, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/admin/osf/osfuser/?username={アカウント(統合管理者)のユーザー名}」を直接入力する

「変更する osf user を選択」ページが表示されること   
アカウント(統合管理者)のユーザー名が表示されること    
※ 画面見出し: 変更する osf user を選択

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/admin/osf/osfuser/?username={integrated_admin_user_name}")

    await expect(page.get_by_role("heading", name="変更する osf user を選択")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name=f"{integrated_admin_user_name}")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「変更する osf user を選択」ページに、アカウント(統合管理者)のユーザー名をクリックする

「osf user を変更」ページが表示されること   
「Is registered」と「Is active」のチェックボックスがチェックされていること   
※ 画面見出し: osf user を変更

In [ ]:
async def _step(page):
    await page.get_by_role("link", name=f"{integrated_admin_user_name}").click()

    await expect(page.get_by_role("heading", name="osf user を変更", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#id_is_registered")).to_be_checked(timeout=transition_timeout)
    await page.locator("#id_is_registered").scroll_into_view_if_needed()
    await expect(page.locator("#id_is_active")).to_be_checked(timeout=transition_timeout)
    await page.locator("#id_is_active").scroll_into_view_if_needed()

await run_pw(_step)

## 「osf user を変更」ページに、「Is registered」のチェックを外し、「保存」ボタンをクリックする

「変更する osf user を選択」ページが表示されること   
「Osf user {アカウント(統合管理者)のユーザー名} を変更しました。」というメッセージが表示されること   
※ 画面見出し: 変更する osf user を選択   
※「保存」ボタンをクリックすると、「Is active」の設定が「Is registered」と同じ状態になること。「Is registered」がチェックされていない場合、「Is active」もチェックされていない状態になること。   
※ 実施による影響：統合管理者が 「クリーンアップ処理」シートNo.5まで無効になること。

In [ ]:
async def _step(page):
    integrated_admin_user_name_update_url = page.url
    await page.locator("#id_is_registered").uncheck()
    await page.get_by_role("button", name="保存", exact=True).click()

    # Save integrated_admin_user_name_update_url to a file
    with open(url_list_file, 'a') as f:
        f.write(f'integrated_admin_user_name_update_url={integrated_admin_user_name_update_url}\n')

    await expect(page.locator(".text-success")).to_contain_text(f'Osf user "{integrated_admin_user_name}" を変更しました。',timeout=transition_timeout)

await run_pw(_step)

## アカウント(統合管理者2)をログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    logout_btn = page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link = page.locator('//*[@href="/account/logout/"]')
    if await logout_btn.count() > 0:
        await logout_btn.click()
    else:
        await logout_link.click()

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アカウント(統合管理者)を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。   

In [ ]:
async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_integrated_admin, idp_username_integrated_admin, idp_password_integrated_admin, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/quota_recalc」を直接入力する	

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること 

In [ ]:
forbidden_permission_status = 403
forbidden_permission_message = "You do not have permission."

async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/quota_recalc")

    await expect(page.get_by_role("heading", name=f'{forbidden_permission_status}')).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/quota_recalc/{ユーザーのGUID}」を直接入力する

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること 

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/quota_recalc/{user_guid}")

    await expect(page.get_by_role("heading", name=f'{forbidden_permission_status}')).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/statistics/test/mail」を直接入力する	

ページにアクセスできないこと    
「You do not have permission.」というテキストが表示されること 

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/statistics/test/mail")

    await expect(page.get_by_role("heading", name=f'{forbidden_permission_status}')).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_text(forbidden_permission_message)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アカウント(統合管理者)をログアウトする

ログアウトが完了すること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    logout_btn = page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')
    logout_link = page.locator('//*[@href="/account/logout/"]')
    if await logout_btn.count() > 0:
        await logout_btn.click()
    else:
        await logout_link.click()

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}